## ANN Parameter Tuning (California Housing Dataset)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
import matplotlib.pyplot as plt

In [ ]:
raw = np.loadtxt('datasets/CaliforniaHousing/cal_housing.data', delimiter=',')
housing = {
    'data': raw[:, :-1],
    'target': raw[:, -1] / 100000.0,
    'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
}
print('Loaded local dataset from datasets/CaliforniaHousing/cal_housing.data')

In [ ]:
print('Housing data shape:', housing['data'].shape)
print('Housing target shape:', housing['target'].shape)

In [ ]:
# train and test split
X_train_full, X_test, y_train_full, y_test = train_test_split(housing['data'], housing['target'], random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

In [ ]:
# scaling the dataset

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)
X_train_full = scaler.fit_transform(X_train_full)

In [ ]:
## build a parameter tuner
## we are tuning the number of layers, number of nodes in hidden layers and the learning rate

def build_model(hp):
    model = keras.models.Sequential()

    # Tune number of hidden layers: 1 to 4
    num_layers = hp.Int('num_layers', 1, 4)

    # Tune number of nodes in hidden layers: 8 to 64 by increment of 4
    for i in range(num_layers):
        units = hp.Int(f'units_{i}', min_value=8, max_value=64, step=4)
        activation = hp.Choice(f'activation_{i}', ['relu', 'tanh'])

        if i == 0:
            model.add(keras.layers.Dense(units=units, activation=activation, input_shape=X_train.shape[1:]))
        else:
            model.add(keras.layers.Dense(units=units, activation=activation))

    model.add(keras.layers.Dense(1))

    model.compile(
        loss='mse',
        optimizer=keras.optimizers.Adam(learning_rate=hp.Float('lr', 1e-5, 1e-2, sampling='log')),
        metrics=[keras.metrics.RootMeanSquaredError()]
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=20,
    overwrite=True
)

In [ ]:
## run tuner
tuner.search(X_train, y_train,
             validation_data=(X_valid, y_valid),
             epochs=20,
             callbacks=[keras.callbacks.EarlyStopping(patience=3)])

In [ ]:
# retreive the best model after tuning
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\nBest Hyperparameters found:")
print(f"Number of hidden layers: {best_hp.get('num_layers')}")
for i in range(best_hp.get('num_layers')):
    print(f" Layer {i+1}: units = {best_hp.get(f'units_{i}')}, activation = {best_hp.get(f'activation_{i}')}")
print(f"Learning rate: {best_hp.get('lr')}")

In [ ]:
# Get the top 3 hyperparameter sets
best_hps_list = tuner.get_best_hyperparameters(num_trials=3)

print("\n=== Top 3 Hyperparameter Configurations ===\n")
for rank, hp in enumerate(best_hps_list, start=1):
    print(f"Rank #{rank}:")
    for param, value in hp.values.items():
        print(f"  {param}: {value}")
    # Build model with this configuration and evaluate
    model = tuner.hypermodel.build(hp)
    eval_result = model.evaluate(X_test, y_test, verbose=0)
    print(f"  Test RMSE: {eval_result[1]:.4f}\n")

In [ ]:
# retrain using the best model
print("\nRetraining best model with more epochs...")
best_model = build_model(best_hp)
history = best_model.fit(
    X_train_full, y_train_full,
    validation_data=(X_valid, y_valid),
    epochs=50,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1, 2, 1)
plt.plot(history.history['root_mean_squared_error'], label='Train RMSE')
plt.plot(history.history['val_root_mean_squared_error'], label='Val RMSE')
plt.xlabel('Epochs')
plt.ylabel('RMSE')
plt.title('Root Mean Squared Error')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.title('Mean Squared Error Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final evaluation
loss, rmse = best_model.evaluate(X_test, y_test)
print("\nFinal best model test RMSE:", rmse)